# Module 1 — Homework (2026 cohort)
**Stock Markets Analytics Zoomcamp 2026**

Introduction and Data Sources. Download financial data from various sources and run simple calculations.

Brief: https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md

## Setup

In [1]:
!pip install yfinance pandas_datareader lxml --quiet

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

## Question 1 — [Index] S&P 500 stocks added to the index

**Which full year (starting from 2020) had the highest number of additions?**

Scrape the Wikipedia S&P 500 list, extract the year each company was added, count additions per year, take the max from 2020 onward.

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                         'AppleWebKit/537.36 (KHTML, like Gecko) '
                         'Chrome/91.0.4472.124 Safari/537.36'}

# fetch with headers, then let pandas parse the HTML tables
resp = requests.get(url, headers=headers)
sp500 = pd.read_html(StringIO(resp.text))[0]   # first table = constituents

# parse the inclusion date and pull out the year
sp500['Date added'] = pd.to_datetime(sp500['Date added'], errors='coerce')
sp500['year_added'] = sp500['Date added'].dt.year

# additions per year from 2020 onward
per_year = sp500.loc[sp500['year_added'] >= 2020, 'year_added'].value_counts().sort_index()
print(per_year)
print('Q1 (year with most additions since 2020):', int(per_year.idxmax()))

year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64
Q1 (year with most additions since 2020): 2025


In [3]:
# Additional: how many current constituents have been in the index > 20 years
cutoff = pd.Timestamp.now() - pd.DateOffset(years=20)
print('In the index > 20 years:', int((sp500['Date added'] < cutoff).sum()))

In the index > 20 years: 224


## Question 2 — [Macro] Indexes YTD (as of 21 Aug 2026)

**How many of the other 10 indexes have a better YTD return than the S&P 500 (1 Jan – 21 Aug 2026)?**

Download daily closes for all indexes, take first vs last available close in the window, compare growth to `^GSPC`.

In [4]:
indexes = {
    '^GSPC':    'US - S&P 500',
    '000001.SS':'China - Shanghai Composite',
    '^HSI':     'Hong Kong - Hang Seng',
    '^AXJO':    'Australia - ASX 200',
    '^NSEI':    'India - Nifty 50',
    '^GSPTSE':  'Canada - TSX Composite',
    '^GDAXI':   'Germany - DAX',
    '^FTSE':    'UK - FTSE 100',
    '^N225':    'Japan - Nikkei 225',
    '^MXX':     'Mexico - IPC',
    '^BVSP':    'Brazil - Ibovespa',
}

# end is exclusive in yfinance -> use 2026-08-22 to include 2026-08-21
data = yf.download(list(indexes.keys()),
                   start='2026-01-01', end='2026-08-22')['Close']

# per-index YTD growth using first and last *available* close (markets differ on holidays)
growth = {}
for tk in indexes:
    s = data[tk].dropna()
    growth[tk] = (s.iloc[-1] / s.iloc[0] - 1) * 100

g = pd.Series(growth).rename('YTD %').sort_values(ascending=False)
print(g.round(2))

sp = g['^GSPC']
better = (g.drop('^GSPC') > sp).sum()
print('\nS&P 500 YTD:', round(sp, 2), '%')
print('Q2 (indexes beating S&P 500):', int(better))

[*********************100%***********************]  10 of 11 completed

^N225        27.36
^GSPTSE      14.86
^GSPC        11.90
^FTSE         8.70
^BVSP         6.54
^GDAXI        6.51
^AXJO         3.79
^MXX          2.48
^HSI         -1.25
000001.SS    -2.94
^NSEI        -7.25
Name: YTD %, dtype: float64

S&P 500 YTD: 11.9 %
Q2 (indexes beating S&P 500): 2


## Question 3 — [Index] S&P 500 market corrections

**Median drawdown (%) of corrections (>= 5% down from a running all-time high), data from 1950.**

For each pair of consecutive all-time-high days, find the trough in between, compute drawdown `(high - low) / high * 100`, keep those >= 5%.

In [5]:
spx = yf.download('^GSPC', start='1950-01-01')['Close']
spx = spx.squeeze().dropna()

# all-time-high days: price >= all previous prices
ath = spx[spx >= spx.cummax()]
ath_dates = list(ath.index)

corrections = []
for i in range(len(ath_dates) - 1):
    peak_date, next_peak = ath_dates[i], ath_dates[i + 1]
    window = spx.loc[peak_date:next_peak]
    high = spx.loc[peak_date]
    low = window.min()
    trough_date = window.idxmin()
    dd = (high - low) / high * 100
    if dd >= 5:
        corrections.append({
            'peak': peak_date,
            'trough': trough_date,
            'drawdown_pct': float(dd),
            'duration_days': (trough_date - peak_date).days,
        })

corr = pd.DataFrame(corrections)
print('Number of corrections (>=5%):', len(corr))
print('Q3 (median drawdown %):', round(corr['drawdown_pct'].median(), 2))

print('\nDrawdown percentiles:')
print(corr['drawdown_pct'].quantile([.25, .5, .75]).round(2))
print('\nDuration percentiles (days):')
print(corr['duration_days'].quantile([.25, .5, .75]).round(0))

print('\nTop 10 by drawdown (sanity check vs the brief):')
print(corr.sort_values('drawdown_pct', ascending=False).head(10).to_string(index=False))

[*********************100%***********************]  1 of 1 completed


Number of corrections (>=5%): 74
Q3 (median drawdown %): 7.99

Drawdown percentiles:
0.25     6.23
0.50     7.99
0.75    14.02
Name: drawdown_pct, dtype: float64

Duration percentiles (days):
0.25    22.0
0.50    40.0
0.75    86.0
Name: duration_days, dtype: float64

Top 10 by drawdown (sanity check vs the brief):
      peak     trough  drawdown_pct  duration_days
2007-10-09 2009-03-09     56.775388            517
2000-03-24 2002-10-09     49.146948            929
1973-01-11 1974-10-03     48.203593            630
1968-11-29 1970-05-26     36.061641            543
2020-02-19 2020-03-23     33.924960             33
1987-08-25 1987-12-04     33.509515            101
1961-12-12 1962-06-26     27.973568            196
1980-11-28 1982-08-12     27.113582            622
2022-01-03 2022-10-12     25.425097            282
1966-02-09 1966-10-07     22.177335            240


## Question 4 — [Stocks] Earnings surprise analysis for AMZN

**Median 2-day return after positive earnings surprises + correlation with surprise magnitude.**

2-day return for a day t (the announcement, "Day 2") = `Close[t+1] / Close[t-1] - 1`.

In [6]:
ticker = 'AMZN'
tk = yf.Ticker(ticker)

# earnings dates with Reported EPS / Surprise(%)
earn = tk.get_earnings_dates(limit=40)
earn = earn.dropna(subset=['Surprise(%)'])   # drop the future entry with no data

prices = yf.download(ticker, start='2019-01-01')['Close'].squeeze().dropna()

# centered 2-day return: Close_Day3 / Close_Day1 - 1  (current day = Day 2)
two_day = prices.shift(-1) / prices.shift(1) - 1
two_day.index = two_day.index.date            # align on calendar date

pos = earn[earn['Surprise(%)'] > 0].copy()
pos['ret_2d'] = [two_day.get(d.date(), np.nan) for d in pos.index]
pos = pos.dropna(subset=['ret_2d'])

print('Positive-surprise events matched:', len(pos))
print('Q4 (median 2-day return, %):', round(pos['ret_2d'].median() * 100, 2))
print('Correlation (surprise vs 2-day return):',
      round(pos['Surprise(%)'].corr(pos['ret_2d']), 3))

[*********************100%***********************]  1 of 1 completed

Positive-surprise events matched: 24
Q4 (median 2-day return, %): 1.26
Correlation (surprise vs 2-day return): 0.313


In [7]:
# Q4 diagnostic
print('rows with Surprise%:', len(earn))
print('positive:', int((earn['Surprise(%)'] > 0).sum()),
      '| negative:', int((earn['Surprise(%)'] < 0).sum()),
      '| zero:', int((earn['Surprise(%)'] == 0).sum()))
print(earn[['Surprise(%)']].sort_index().to_string())

rows with Surprise%: 49
positive: 36 | negative: 13 | zero: 0
                           Surprise(%)
Earnings Date                         
2014-07-24 16:00:00-04:00       -83.42
2014-10-23 16:00:00-04:00       -23.73
2015-01-29 16:00:00-05:00       152.53
2015-04-23 16:00:00-04:00       -12.15
2015-07-23 16:00:00-04:00       240.32
2015-10-22 16:00:00-04:00       231.17
2016-01-28 16:00:00-05:00       -37.27
2016-04-28 16:00:00-04:00        76.92
2016-07-28 16:00:00-04:00        63.72
2016-10-27 16:00:00-04:00       -35.74
2017-02-02 16:00:00-05:00         9.70
2017-04-27 16:00:00-04:00        34.30
2017-07-27 16:00:00-04:00       -71.37
2017-10-26 16:00:00-04:00      3900.00
2018-02-01 16:00:00-05:00        16.43
2018-04-26 16:00:00-04:00       165.81
2018-07-26 16:00:00-04:00        99.34
2018-10-25 16:00:00-04:00        86.22
2019-01-31 16:00:00-05:00         9.54
2019-04-25 16:00:00-04:00        51.64
2019-07-25 16:00:00-04:00        -5.83
2019-10-24 16:00:00-04:00        -5.79
20

In [8]:
# keep only the window the brief expects: from 2020-10-29 onward, ~25 rows
earn = earn[earn.index >= '2020-10-29']
print('rows in brief window:', len(earn))          # expect ~25 (24 with Surprise%)

pos = earn[earn['Surprise(%)'] > 0].copy()
pos['ret_2d'] = [two_day.get(d.date(), np.nan) for d in pos.index]
pos = pos.dropna(subset=['ret_2d'])

print('Positive-surprise events matched:', len(pos))
print('Q4 (median 2-day return, %):', round(pos['ret_2d'].median() * 100, 2))
print('Correlation (surprise vs 2-day return):',
      round(pos['Surprise(%)'].corr(pos['ret_2d']), 3))

rows in brief window: 24
Positive-surprise events matched: 20
Q4 (median 2-day return, %): 0.35
Correlation (surprise vs 2-day return): 0.331


## Question 5 — [Exploratory, optional] Capstone project idea

**Capstone: Multilingual factual reliability of public LLMs on verifiable financial facts**

I want to extend the methodology of my main project (VeriHub — auditing how public
LLMs represent organisations) into a domain with objective, free ground truth: factual
questions about S&P 500 companies. Instead of predicting price direction, my ML target
is predicting *when an LLM answer will be wrong*.

- **Universe:** current S&P 500 constituents (from the Wikipedia list), with a focus on
  a stratified sample across market cap, sector and liquidity.
- **Languages:** English, French, Dutch (the multilingual angle central to VeriHub).
- **Task:** ask several public LLMs verifiable questions per ticker — latest dividend,
  P/E, last earnings date, 52-week high, current index membership — and score each
  answer against ground truth from Yahoo Finance / FRED / Wikipedia.
- **ML model:** a binary classifier predicting whether a given (model, question, ticker,
  language) tuple produces a factual error. Candidate features: realized volatility of
  the stock, recency of the underlying fact (days since last earnings / index change),
  ticker popularity (market cap, average volume, news coverage), question type, and
  query language.
- **Horizon / output:** a per-question "hallucination risk score" and an analysis of
  which conditions (fast-moving facts, low-coverage tickers, non-English queries) drive
  errors.

**Why this fits me:** it reuses VeriHub's audit design (I already ran a multilingual
audit of four assistants on 74 real questions), but on a domain where the correct answer
is cheap and unambiguous — an ideal benchmark before applying the same layer to public
organisations.

## Question 6 — [Exploratory, optional] New metrics

**Additional metrics / time series for the project, and why each matters**

1. **Realized volatility** (from OHLCV). Proxy for how fast a company's facts move —
   high-volatility names have quickly-stale prices/levels, so LLMs are likelier to be
   out of date. Retrieve with yfinance, compute rolling std of log-returns.

2. **Market cap & average trading volume** (liquidity / popularity). Well-known, heavily
   traded companies appear more often in training data, so LLMs likely represent them
   better; this is a strong candidate feature for error prediction. From
   `yf.Ticker(t).info` (marketCap, averageVolume).

3. **Earnings dates & surprise** (event recency). Questions asked right after an earnings
   release concern very fresh facts — a likely error hotspot. From
   `yf.Ticker(t).get_earnings_dates()` (reused from Q4).

4. **Index membership changes** (fact recency). LLMs lag on recent S&P 500 additions /
   removals. The "Selected changes to the S&P 500" table on Wikipedia gives the timeline;
   scrape it with pandas.read_html.

5. **News coverage volume** (representation density). More published text about a company
   should mean better LLM coverage; a coverage count is a natural feature. Retrieve via a
   news/RSS API (e.g. GDELT or a finance-news feed) aggregated per ticker per period.

These aren't a final feature set — they show how I'd translate the project description
into concrete, retrievable data requests.

In [9]:
import yfinance as yf
import numpy as np

t = 'AAPL'
px = yf.download(t, start='2023-01-01')['Close'].squeeze()

# 1) realized volatility (annualised 1-month)
ret = np.log(px / px.shift(1))
realized_vol = (ret.rolling(21).std() * np.sqrt(252)).dropna()

# 2) popularity / liquidity features
info = yf.Ticker(t).info
print('market cap:', info.get('marketCap'))
print('avg volume:', info.get('averageVolume'))
print('latest realized vol:', round(realized_vol.iloc[-1], 3))

[*********************100%***********************]  1 of 1 completed


market cap: 4877668319232
avg volume: 54055456
latest realized vol: 0.232
